## **Install the required modules**

In [1]:
!pip install -qU \
accelerate==0.31.0 \
peft==0.11.1 \
bitsandbytes==0.43.1 \
transformers==4.41.2 \
trl==0.9.4 \
sentencepiece==0.2.0 \
triton==3.1.0 \
torchvision


**Restart the session after all the modules are installed.**
Restart the session after all the modules are installed.

##**Import the required Libraries**

In [1]:
# Standard Deep Learning and Hardware Framework
import torch
import transformers
import accelerate
import peft
import trl
import torchvision

# Hugging Face Datasets
from datasets import load_dataset

# Hugging Face Transformers & Pipelines
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline, TrainingArguments

# PEFT (Parameter-Efficient Fine-Tuning) / LoRA Tools
from peft import AutoPeftModelForCausalLM, LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model

# TRL (Transformer Reinforcement Learning) Trainers & Configs
from trl import SFTTrainer, DPOConfig, DPOTrainer

# Utilities
from google.colab import drive
import os
import shutil

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("torchvision:", torchvision.__version__)
print("accelerate:", accelerate.__version__)

torch: 2.5.1+cu124
transformers: 4.41.2
peft: 0.11.1
trl: 0.9.4
torchvision: 0.20.1+cu124
accelerate: 0.31.0


###**Mount Google Drive and set it up for storing the artifacts**

In [ ]:
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Paths for intermediate checkpoints and final merged model
checkpoint_drive_path = '/content/drive/MyDrive/Fine_Tuned_Models/tiny_llama_mcq_checkpoints'
adapter_drive_path = "/content/drive/MyDrive/Fine_Tuned_Models/Tinyllama-1.1B-mcq-adapter"
merged_model_drive_path = "/content/drive/MyDrive/Fine_Tuned_Models/Tinyllama-1.1B-mcq"

# Create folders if they do not exist
os.makedirs(checkpoint_drive_path, exist_ok=True)
os.makedirs(adapter_drive_path, exist_ok=True)
os.makedirs(merged_model_drive_path, exist_ok=True)

##**Tiny LLama 1.1 Biliion Chat Model - Version 1.0**

###**Before Fine Tuning**

In [7]:
chat_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
#chat_model_name = "meta-llama/Llama-3.1-8B-Instruct"

chat_tokenizer = AutoTokenizer.from_pretrained(chat_model_name)
chat_model = AutoModelForCausalLM.from_pretrained(chat_model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [8]:
mcq_generator_chat = pipeline(
"text-generation",
model=chat_model,
tokenizer=chat_tokenizer,
return_full_text=False,
max_new_tokens=150,
do_sample=True,
temperature=0.7
)

###**First Try**

In [9]:
context = """
Photosynthesis is a biological process used by plants, algae, and certain bacteria to convert light energy into chemical energy stored in glucose. It occurs mainly in the chloroplasts of plant cells using chlorophyll pigments.
"""
target_answer = "chloroplasts"

messages = [
    {
        "role": "system",
        "content": "You are an expert educational assessment AI that generates a clear, high-quality multiple-choice question based strictly on given a context and target answer."
    },
    {
        "role": "user",
        "content": f"Context: {context}\nTarget Answer: {target_answer}\nGenerate a question from the given context where the target answer is the correct answer. Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\nThe output should be in the form\nQuestion:\nAnswer:"
    }
]

In [10]:
output = mcq_generator_chat(messages)
output

[{'generated_text': 'Question: Which of the following is a type of chloroplast in plants?\nAnswer: Answer: chloroplasts'}]

In [11]:
output[0]['generated_text']

'Question: Which of the following is a type of chloroplast in plants?\nAnswer: Answer: chloroplasts'

###**Second Try**

In [ ]:
context = """
In operating systems, a deadlock is a situation where a set of processes are blocked because each process is holding a resource and waiting for another resource held by some other process. The Banker's algorithm is a classical resource allocation algorithm developed by Edsger Dijkstra to avoid deadlocks.
"""
target_answer = "Banker's algorithm"

messages = [
    {
        "role": "system",
        "content": "You are an expert educational assessment AI that generates a clear, high-quality multiple-choice question based strictly on given a context and target answer."
    },
    {
        "role": "user",
        "content": f"Context: {context}\nTarget Answer: {target_answer}\nGenerate a question from the given context where the target answer is the correct answer. Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\nThe output should be in the form\nQuestion:\nAnswer:"
    }
]

In [ ]:
output = mcq_generator_chat(messages)
output

[{'generated_text': "Question: In what context does Banker's algorithm avoid deadlocks in operating systems?\n\nAnswer: Banker's algorithm is a classical resource allocation algorithm developed by Edsger Dijkstra to avoid deadlocks in operating systems."}]

In [ ]:
output[0]['generated_text']

"Question: In what context does Banker's algorithm avoid deadlocks in operating systems?\n\nAnswer: Banker's algorithm is a classical resource allocation algorithm developed by Edsger Dijkstra to avoid deadlocks in operating systems."

**So as we see the chat model generates the multiple choice question from a context and target answer. But the questions generated do not elicit the answer given properly. So it needs to be trained to make the output optimal.**

##**Checking the Training Data**

###**Squad V2 Dataset**

In [2]:
train_dataset = load_dataset('rajpurkar/squad_v2', split='train').shuffle(seed=42).select(range(15000))

df = train_dataset.to_pandas()
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

,id,title,context,question,answers
0,56e0f3907aa994140058e80a,Canon_law,The Roman Catholic Church canon law also inclu...,What term characterizes the intersection of th...,"{'text': ['full union'], 'answer_start': [104]}"
1,571adcf932177014007e9f56,Athanasius_of_Alexandria,Alexandria was the most important trade center...,What was Alexandria known for?,"{'text': ['important trade center'], 'answer_s..."
2,57325b9fe99e3014001e670c,Jehovah%27s_Witnesses,Former members Heather and Gary Botting compar...,How do the leaders of the Jehovah's Witnesses ...,{'text': ['disparaging individual decision-mak...
3,5728d8be4b864d1900164f6b,Estonia,"Historically, the cuisine of Estonia has been ...",What are the most common foods in Estonia?,"{'text': ['black bread, pork, potatoes, and da..."
4,56f6f5e1711bf01900a44898,Classical_music,Many of the instruments used to perform mediev...,What was the medieval flute made from?,"{'text': ['wood'], 'answer_start': [126]}"


In [3]:
train_dataset

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 15000
})

###**Mapping Function**

In [4]:
def format_prompt(example, tokenizer):
    # Extract context and target answer safely (handling SQuAD v2 unanswerable questions)
    context = example["context"]
    answers = example.get("answers", {})

    if answers and len(answers.get("text", [])) > 0:
        target_answer = answers["text"][0]
    else:
        target_answer = "None"

    question = example.get("question", "")

    # Construct the conversational messages including system, user, and assistant turns
    messages = [
        {
            "role": "system",
            "content": "You are an expert educational assessment AI that generates a clear, high-quality question based strictly on a given context and target answer."
        },
        {
            "role": "user",
            "content": f"Context: {context}\nTarget Answer: {target_answer}\nGenerate a question from the given context where the target answer is the correct answer. Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\nThe output should be in the form\nQuestion:\nAnswer:"
        },
        {
            "role": "assistant",
            "content": f"Question: {question}\nAnswer: {target_answer}"
        }
    ]

    # Apply TinyLlama's chat template to format the prompt for training
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": formatted_text}

##**Model Quantization**

In [5]:
# 4-bit quantization configuration - Q in QLoRA
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,  # Use 4-bit precision model loading
  bnb_4bit_quant_type="nf4",  # Quantization type
  bnb_4bit_compute_dtype=torch.float16,  # Compute dtype
  bnb_4bit_use_double_quant=True,  # Apply nested quantization
)

In [12]:
# Load the model to train on the GPU
model = AutoModelForCausalLM.from_pretrained(
  chat_model_name,
  device_map="auto",

  # Leave this out for regular SFT
  quantization_config=bnb_config,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

In [13]:
# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(chat_model_name, trust_remote_code=True)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"

In [14]:
tokenized_train_dataset = train_dataset.map(lambda x: format_prompt(x,tokenizer))

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

In [15]:
tokenized_train_dataset

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers', 'text'],
    num_rows: 15000
})

In [16]:
print(tokenized_train_dataset["text"][2576])

<|system|>
You are an expert educational assessment AI that generates a clear, high-quality question based strictly on a given context and target answer.</s>
<|user|>
Context: Tristan da Cunha /ˈtrɪstən də ˈkuːnjə/, colloquially Tristan, is both a remote group of volcanic islands in the south Atlantic Ocean and the main island of that group. It is the most remote inhabited archipelago in the world, lying 2,000 kilometres (1,200 mi) from the nearest inhabited land, Saint Helena, 2,400 kilometres (1,500 mi) from the nearest continental land, South Africa, and 3,360 kilometres (2,090 mi) from South America. The territory consists of the main island, also named Tristan da Cunha, which has a north–south length of 11.27 kilometres (7.00 mi) and has an area of 98 square kilometres (38 sq mi), along with the smaller, uninhabited Nightingale Islands and the wildlife reserves of Inaccessible and Gough Islands.
Target Answer: None
Generate a question from the given context where the target answer

###**LoRA Configuration**

In [18]:
# Prepare LoRA Configuration
peft_config = LoraConfig(
  lora_alpha=32,  # LoRA Scaling
  lora_dropout=0.1,  # Dropout for LoRA Layers
  r=64,  # Rank
  bias="none",
  task_type="CAUSAL_LM",
  target_modules=  # Layers to target
  ["q_proj", "v_proj", "k_proj", "o_proj"]
  )

In [19]:
# Prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

###**Training Configuration**

In [20]:
# Training arguments
training_arguments = TrainingArguments(
  output_dir=checkpoint_drive_path,
  per_device_train_batch_size=8,
  gradient_accumulation_steps=4, #Effective batch size is now 32 (8x4)
  optim="paged_adamw_32bit",
  learning_rate=2e-4,
  #report_to="none", #Turn off wandb reporting
  lr_scheduler_type="cosine",
  num_train_epochs=1,
  logging_steps=10,
  fp16=True,
  gradient_checkpointing=True,
  # Checkpoint Auto-Saving (Protects against sudden Colab disconnections)
  save_strategy="steps",
  save_steps=100, # Saves progress to Google Drive every 100 steps
  save_total_limit=2, # Keeps the 2 latest saves to avoid filling Drive space

  )

In [21]:
# Set supervised fine-tuning parameters
trainer = SFTTrainer(
  model=model,
  train_dataset=tokenized_train_dataset,
  dataset_text_field="text",
  tokenizer=tokenizer,
  args=training_arguments,
  max_seq_length=512,
  # Leave this out for regular SFT
  peft_config=peft_config,
  )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1965: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:269: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:307: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:397: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:477: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [22]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.045200
20,1.473200
30,1.352300
40,1.349900
50,1.313200
60,1.346800
70,1.336200
80,1.343200
90,1.330700
100,1.318400


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `

TrainOutput(global_step=468, training_loss=1.326313130875938, metrics={'train_runtime': 4519.0302, 'train_samples_per_second': 3.319, 'train_steps_per_second': 0.104, 'total_flos': 4.201922628462182e+16, 'train_loss': 1.326313130875938, 'epoch': 0.9984})

###**WARNING! If Training fails run this cell. Else skip this cell.**

Resume Training from a particular Checkpoint. This looks into our Google Drive and loads the exact state from step XXX. Replace XXX with the correct checkpoint.

In [ ]:
checkpoint_dir = os.path.join(checkpoint_drive_path, "checkpoint-XXX")
trainer.train(resume_from_checkpoint=checkpoint_dir)

###**Save the Adapter Files to Google Drive**

Save the final trained LoRA adapters to Google Collab folders locally first

In [27]:

trainer.model.save_pretrained("TinyLlama-1.1B-mcq")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Then Copy to Google Drive

In [33]:
adapter_local_path = "/content/TinyLlama-1.1B-mcq"

shutil.copytree(adapter_local_path, adapter_drive_path, dirs_exist_ok=True)

print(f"LoRA adapter files successfully copied to: {adapter_drive_path}")

LoRA adapter files successfully copied to: /content/drive/MyDrive/Fine_Tuned_Models/Tinyllama-1.1B-mcq-adapter


###**Merge Adapter**

In [28]:
# Free up VRAM before loading the merge tool
del model
del trainer
torch.cuda.empty_cache()

In [29]:
model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-mcq",
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Merge LoRA and base model
merged_model = model.merge_and_unload()

###**Save and Download the Merged Model to Google Drive**

In [32]:
# Save directly to your Google Drive
merged_model.save_pretrained(merged_model_drive_path)
tokenizer.save_pretrained(merged_model_drive_path)
print(f"Model and tokenizer safely saved to your Google Drive at: {merged_model_drive_path}")

Model and tokenizer safely saved to your Google Drive at: /content/drive/MyDrive/Fine_Tuned_Models/Tinyllama-1.1B-mcq


###**Using the Merged Model**

In [30]:
# Use our predefined prompt template
context = """
In operating systems, a deadlock is a situation where a set of processes are blocked because each process is holding a resource and waiting for another resource held by some other process. The Banker's algorithm is a classical resource allocation algorithm developed by Edsger Dijkstra to avoid deadlocks.
"""
target_answer = "Banker's algorithm"

messages = [
    {
        "role": "system",
        "content": "You are an expert educational assessment AI that generates a clear, high-quality multiple-choice question based strictly on given a context and target answer."
    },
    {
        "role": "user",
        "content": f"Context: {context}\nTarget Answer: {target_answer}\nGenerate a question from the given context where the target answer is the correct answer. Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\nThe output should be in the form\nQuestion:\nAnswer:"
    }
]

# Run our instruction-tuned model
pipe = pipeline(
    task="text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.7
    )
output = pipe(messages)
output

[{'generated_text': "Question: What algorithm is used to avoid deadlocks?\nAnswer: Banker's algorithm"}]

In [31]:
output[0]['generated_text']

"Question: What algorithm is used to avoid deadlocks?\nAnswer: Banker's algorithm"